# Show the Path of the Warm Water Inflow at 79NG

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from tools.visualization import cm

## Define the Connected Set Finder

In [ ]:
# Part of this class was written with the help of ChatGPT 3.5 (18 April 2024)
class ConnectedSetFinder:
    """Class to find connected sets of valid grid points."""

    # Directions in which grid points are considered as connected
    NEIGHBOURS = [(1, 0), (0, 1), (-1, 0), (0, -1)]

    def __init__(self, mask: np.ndarray):
        """Initialize a ConnectionFinder object on the given mask array.

        The mask is True on valid and False on invalid grid points.
        """
        self.valid = mask
        self.ni, self.nj = mask.shape

    def dfs(self, i: int, j: int, connected_indices: list):
        """Perform a depth-first search to find connected indices."""
        if self.valid[i, j] and (i, j) not in self.visited:
            self.visited.add((i, j))
            connected_indices.append((i, j))
            for di, dj in self.NEIGHBOURS:
                if 0 <= i + di < self.ni and 0 <= j + dj < self.nj:
                    self.dfs(i + di, j + dj, connected_indices)

    def find_all_connections(self):
        """Get the indices of all connected sets of valid grid points."""
        connections = []
        self.visited = set()
        for i in range(self.ni):
            for j in range(self.nj):
                connected_indices = []
                self.dfs(i, j, connected_indices)
                if connected_indices:
                    connections.append(connected_indices)
        return connections

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc").set_coords("zc")

# Convert single time value from coordinate to attribute
assert ds.time.size == 1
time_value = ds.time.data[0]
ds = ds.squeeze(dim="time", drop=True)
ds.attrs["time"] = time_value

# Add (missing) geometry variables to the dataset
with xr.open_dataset("output/out_geometry_2d.nc") as geo:
    for var in geo.variables:
        if var in ds:
            print(var, "exists in dataset and geometry.")
            # Check that the data matches between dataset and geometry
            xr.testing.assert_equal(ds[var], geo[var])
        else:
            ds[var] = geo[var]

# Compute the masks
ds["mask"] = ds.bathymetry.notnull()
ds.mask.attrs = {"long_name": "ocean mask"}
ds["ice_mask"] = ds.glIceD > 0
ds.ice_mask.attrs = {"long_name": "ice mask"}

# Compute the flow speed
assert ds.velx3d.units == ds.vely3d.units, "units of u3d and v3d differ"
ds["vel3d"] = np.sqrt(ds.velx3d**2 + ds.vely3d**2)
ds.vel3d.attrs = {"long_name": "absolute value of the global velocity (3D)", "units": ds.velx3d.units}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

# Update the units for better labels with xarray
ds.latc.attrs.update({"units": "°N"})
ds.lonc.attrs.update({"units": "°E"})
ds.temp.attrs.update({"units": "°C"})

ds

## Load the inflow path

This path has been constructed in the notebook `Inflow_analysis_path.ipynb` based on model topography and results.

In [ ]:
path = xr.open_dataset("data/path_inflow.nc")
# Put the start point in flow direction at kilometer 0
path["dist"] = ("dist", (path.dist - path.dist.max()).data, path.dist.attrs)
path.plot.scatter(x="lonc", y="latc", hue="dist")
path

## Extract data along the path

In [ ]:
inflow = ds.sel(path)
inflow

## Select area of interest

In [ ]:
fjord = ds.sel(lonc=slice(-19), latc=slice(79.25, 79.85))
fjord.bathymetry.plot(figsize=(8, 5))
fjord.glIceD.plot.contour(cmap=cmo.ice_r, add_colorbar=True)
plt.gca().set_aspect(grille_carree)
fjord

## Select inflowing warm water

We consider all waters with a temperature above 1 °C and a velocity ($|\vec{u}|$) of at least 0.05 m/s.
In the cavity, i.e., to the west of 19.5°W, we consider only westward flowing water ($u < 0$).

In [ ]:
is_inflow = (
    (fjord.temp > 1)
    & (fjord.vel3d >= 0.05)
    & ((fjord.velx3d < 0) | (fjord.lonc >= -19.5))
)
inflow_mask = is_inflow.any("level")
inflow_mask.plot()
plt.title("Warm water flowing westward")
plt.gca().set_aspect(grille_carree)

## Limit the selection to a continuous inflow

We select the largest set of water columns in the warm water inflow.
We do not check explicitly that the vertical levels are also continuous, but this is later confirmed by the result inside the cavity.

In [ ]:
# Select the largest connected set of inflow points
connected_sets = ConnectedSetFinder(inflow_mask).find_all_connections()
print(f"Found {len(connected_sets)} connected sets of valid grid points.")
connection = connected_sets[np.argmax([len(connection) for connection in connected_sets])]
print(f"Selected the largest connected set, containing {len(connection)} grid points.")

# Mask out everything except for the selected connection
assert np.all(inflow_mask.data[*np.transpose(connection)]), "not all points in the connected set are valid"
inflow_mask.plot(cmap="Greys", alpha=1/2, add_colorbar=False)
inflow_mask.data[...] = False
inflow_mask.data[*np.transpose(connection)] = True
inflow_mask.plot(cmap="Greys", alpha=1/2, add_colorbar=False)
plt.gca().set_aspect(grille_carree)
print("The selected connection is shown darker.")

## Limit the selection to an inflow of at least 5 m thickness

This removes 8 grid cells with negligible inflow volume from the selected area, to obtain a cleaner outline and more representative averages.

In [ ]:
inflow_thickness = fjord.h.where(is_inflow).sum("level")
inflow_thickness.where(inflow_mask, drop=True).plot()
plt.gca().set(aspect=grille_carree, title="Plume thickness [m]")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=200)
print(f"Cleaning the inflow mask from {(inflow_mask & (inflow_thickness < 5)).sum():d} water columns with less than 5 m of plume thickness.")
inflow_mask.plot(alpha=1/2, add_colorbar=False)
inflow_mask &= (inflow_thickness >= 5)
inflow_mask.plot(cmap="Greys", alpha=1/2, add_colorbar=False)
ax.set(title="Yellow points were removed from the plume mask", aspect=grille_carree);

## Compute thickness and vertical extent of the inflow

The thickness of the inflow has just been computed, so we only need to apply the updated mask.

In [ ]:
inflow_thickness = inflow_thickness.where(inflow_mask, drop=True)
inflow_thickness.plot()
plt.gca().set(aspect=grille_carree, title="Thickness of the inflow");

In [ ]:
inflow_zmax = (fjord.zc + fjord.h / 2).where(is_inflow).max("level").where(inflow_mask, drop=True)
inflow_zmax.plot()
plt.gca().set(aspect=grille_carree, title="Upper interface of the inflow");

In [ ]:
inflow_zmin = (fjord.zc - fjord.h / 2).where(is_inflow).min("level").where(inflow_mask, drop=True)
inflow_zmin.plot()
plt.gca().set(aspect=grille_carree, title="Lower interface of the inflow");

If the lower plume interface plus the plume thickness equals the upper plume interface, then the plume is a homogeneous volume.
Outside the cavity, this is not always the case, but we check in the next cell that this holds west of 19.5°W, i.e., downstream of the sill.

In [ ]:
assert np.allclose(
    (inflow_zmin + inflow_thickness).sel(lonc=slice(-19.5)),
    inflow_zmax.sel(lonc=slice(-19.5)),
    equal_nan=True,
), "inflow plume has holes inside the cavity"

## Analyse the plume

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
(fjord.bathymetry + inflow_zmin).plot(ax=ax, vmin=0, cmap=cmo.amp)
(fjord.bathymetry + inflow_zmin).plot.contour(ax=ax, levels=[0.01, 1, 10, 100], cmap=cmo.gray, linewidths=1)
ax.plot([], [], "0.5", lw=1, label="plume < 0.1, 1, 10, 100 m above ground")
ax.legend(loc="lower left")
ax.set(title="Height of the lower plume interface above ground [m]", aspect=grille_carree);

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
(fjord.eta - inflow_zmax).plot(ax=ax, cmap=cmo.amp_r)
(fjord.eta - inflow_zmax).plot.contour(ax=ax, levels=[100])
ax.set(title="Depth of the upper plume interface below the ice/sea level [m]", aspect=grille_carree);

In [ ]:
j, i = np.where(fjord.eta - inflow_zmax < 100)
print("Coordinates of the inflow being less than 100 m from the ice:")
inflow_zmax.latc.data[j], inflow_zmax.lonc.data[i]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
fjord.temp.where(is_inflow).weighted(fjord.h.fillna(0)).mean("level").where(inflow_mask, drop=True).plot(cmap=cmo.thermal, vmax=1.8)
ax.set(title="Mean temperature", aspect=grille_carree);

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
fjord.temp.where(is_inflow).max("level").where(inflow_mask, drop=True).plot(cmap=cmo.thermal, vmax=1.8)
ax.set(title="Max temperature", aspect=grille_carree);

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
fjord.vel3d.where(is_inflow).weighted(fjord.h.fillna(0)).mean("level").where(inflow_mask, drop=True).plot(cmap=cmo.amp, vmax=0.3)
ax.set(title="Mean speed", aspect=grille_carree);

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
fjord.vel3d.where(is_inflow).max("level").where(inflow_mask, drop=True).plot(cmap=cmo.amp, vmax=0.3)
ax.set(title="Max speed", aspect=grille_carree);

## Compute figure data

The temperature and velocity are vertically averaged over the inflow volume.
In addition, the velocity is coarsend to have one quiver arrow for every 3×3 grid points.
Each arrow represents the inflow velocity averaged over the surrounding 3×3 grid points.

In [ ]:
inflow_T = fjord.temp.weighted(fjord.h.fillna(0) * is_inflow).mean("level").where(inflow_mask, drop=True)
inflow_U = fjord.velx3d.weighted(fjord.h.fillna(0) * is_inflow).mean("level").where(inflow_mask, drop=True)
inflow_V = fjord.vely3d.weighted(fjord.h.fillna(0) * is_inflow).mean("level").where(inflow_mask, drop=True)

# The coarsen method of DataArray allows doing the average and the selection in one step,
# but the result contains values outside the inflow mask.  To apply the inflow mask exactly,
# we need to first compute the averages, then apply the mask, then reduce the grid size.
step = 3
inflow_U = inflow_U.rolling(lonc=step, latc=step, min_periods=1, center=True).mean()
inflow_V = inflow_V.rolling(lonc=step, latc=step, min_periods=1, center=True).mean()
inflow_U = inflow_U.where(inflow_mask).isel(lonc=slice(0, None, step), latc=slice(0, None, step))
inflow_V = inflow_V.where(inflow_mask).isel(lonc=slice(0, None, step), latc=slice(0, None, step))
print(f"Note: arrows are averaged over {step}×{step} grid points.")

fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
inflow_T.plot(ax=ax, cmap=cmo.thermal, vmin=1, vmax=2)
ax.quiver(inflow_U.lonc, inflow_U.latc, inflow_U, inflow_V, color="0.8")
ax.set(title="Mean temperature and velocity", aspect=grille_carree);

## Create the figure

In [ ]:
color_inflow = "#ffa040"
alpha_inflow = 0.6
cmap_mask = ListedColormap(["w", color_inflow], "inflow_mask")
cmap_mask

In [ ]:
fig, axs = plt.subplot_mosaic(
    "aaa\naaa\nbbb\nbbb\ncde", constrained_layout=True, figsize=(12*cm, 15.5*cm), dpi=200
)

ax = axs["a"]
# Show the bathymetry
levels = np.arange(0, 900, 100)
im = fjord.bathymetry.plot.contourf(ax=ax, levels=levels, cmap=cmo.deep)
im.colorbar.set_ticks(levels[::2])
im.colorbar.set_label("bathymetry in m")
# Reverse colorbar to have greater depths at the bottom
im.colorbar.ax.invert_yaxis()
# Show the inflow
inflow_mask.where(inflow_mask).plot.contourf(ax=ax, levels=[0, 0.5, 1], cmap=cmap_mask, alpha=alpha_inflow, add_colorbar=False)
# Show and annotate the ice thickness contours
color = "w"
(-fjord.eta).where(fjord.ice_mask, 0).where(fjord.mask).plot.contour(ax=ax, levels=np.arange(0, 800, 100), colors=color, linewidths=1)
ax.text(-21.00, 79.505, "ice draft\n200 m", ha="right", size="small", color=color)
ax.text(-20.32, 79.505, "ice draft\n100 m", ha="right", size="small", color=color)
# Show and annotate the calving front
color = "darkblue"
ds.glIceD.plot.contour(ax=ax, levels=[0], colors=color, linewidths=1)
ax.annotate(
    "calving\nfront", (-19.32, 79.695), (-19.42, 79.71),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 1, "relpos": (1, 0.5)},
    color=color, ha="right", size="small",
)
# Annotate the sill
color = "0.3"
i_sill = inflow.bathymetry.argmin()
print(f"The sill is at {inflow.latc[i_sill]:.5f}°N, {-inflow.lonc[i_sill]:.5f}°W, {inflow.bathymetry[i_sill]:.2f} m depth.")
ax.annotate(
    "sill", (inflow.lonc[i_sill], inflow.latc[i_sill]), (-19.17, 79.585),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 0, "relpos": (0, 0.5)},
    color=color, size="small",
)
# Show and annotate the inflow path
color = "darkred"
ax.plot(path.lonc, path.latc, ":", color=color)
ax.annotate(
    "transect", (inflow.lonc[5], inflow.latc[5]), (inflow.lonc[5], 79.52),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 0},
    color=color, size="small", ha="center",
)
# Annotate the plume detachment
color = "k"
ax.annotate(
    "detachment\nof the inflow", (-20.22, 79.675), (-19.95, 79.665),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 0, "relpos": (0, 0.5)},
    color=color, size="small", va="center",
)
# Annotate the ice approach
color = "#dd3300"
ax.annotate(
    "warm inflow\nwithin 100 m of\nthe ice tongue", (-21.075, 79.575), (-21.48, 79.67),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 0, "relpos": (0.5, 0)},
    color=color, size="small",
)

ax.set_title("Warm water inflow into the 79NG cavity")

ax = axs["b"]
# Show land in gray (background color) and model domain in white where there are no data
fjord.mask.where(fjord.mask).plot(ax=ax, vmax=2, cmap="Greys", add_colorbar=False)
# Show the temperature
im = inflow_T.plot(ax=ax, cmap=cmo.thermal, vmin=1, vmax=2)
im.colorbar.set_ticks(np.arange(1, 2.1, 0.1), minor=True)
im.colorbar.set_label("temperature in °C")
# Show the velocity
Q_plot = ax.quiver(inflow_U.lonc, inflow_U.latc, inflow_U, inflow_V, color="0.9")
ax.quiverkey(Q_plot, -19.73, 79.705, -0.3, "0.3 m s$^{-1}$", coordinates="data", labelsep=0.02, fontproperties={"size": "small"}, color="k", linewidth=0.1)
# Annotate plume cooling
color = "k"
ax.annotate(
    "cooling", (-19.6, 79.57), (-19.7, 79.525),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 3},
    color=color, size="small", ha="center",
)
# Mark the location of the vertical profiles
lon, lat = -20.9, 79.583
ax.annotate(
    "profiles in (c–e)", (lon, lat), (-20.7, 79.525),
    arrowprops={"arrowstyle": "->", "color": color, "shrinkA": 0, "shrinkB": 3},
    color=color, size="small", ha="center",
)

kwargs = dict(
    va="top", size="small", clip_on=True,
    bbox={"boxstyle": "square", "facecolor": "w", "alpha": 2/3, "edgecolor": "none"},
)
for letter in "ab":
    ax = axs[letter]
    ax.text(0.015, 0.976, f"({letter})", transform=ax.transAxes, **kwargs)
    ax.set(xlabel="", ylabel="", xlim=(-21.5, -19), ylim=(79.5, 79.75), xticks=[-21, -20, -19], yticks=[79.5, 79.6, 79.7], aspect=grille_carree, facecolor="0.8")
    ax.xaxis.set_major_formatter((lambda x, pos: f"{-x}°W") if letter == "b" else "")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")


profile = fjord.sel(lonc=lon, latc=lat, method="nearest")
profile_is_inflow = is_inflow.sel(lonc=lon, latc=lat, method="nearest")
# Get the z-coordinates of the two velocity minima
z_min_1 = profile.zc[profile.vel3d.where(profile.zc < -500).argmin()]
z_min_2 = profile.zc[profile.vel3d.where(profile.zc > -500).argmin()]
kwargsl = dict(linestyle="--", linewidth=1, color="0.3")  # kwargs for lines
kwargst = dict(va="center", size="small", in_layout=False) # kwargs for text
titles = {
    "temp": "$T$ in °C",
    "salt": "$S$ in g kg$^{-1}$",
    "vel3d": r"$|\vec u|$ in m s$^{-1}$",
}
for letter, var in zip("cde", ["temp", "salt", "vel3d"]):
    ax = axs[letter]
    profile[var].plot(ax=ax, y="zc")
    vmin = 0 if var == "vel3d" else profile[var].min()
    ax.fill_betweenx(profile.zc, vmin, profile[var], profile_is_inflow, alpha=alpha_inflow, color=color_inflow, edgecolor="none")
    ax.axhline(z_min_1, **kwargsl)
    ax.axhline(z_min_2, **kwargsl)
    ax.set(xlim=vmin, xlabel="", yticks=[-800, -600, -400, -200], title="")
    ax.set_title(f"({letter}) {titles[var]}", loc="left", size="small")
    if var == "salt":
        ax.set_xticks(np.arange(33.8, 34.6, 0.4))
    elif var == "temp":
        ax.set_xticks([0, 1, 2])
        ax.axvline(1, **kwargsl)
    elif var == "vel3d":
        ticks = [0, 0.05, 0.1]
        ax.set_xticks(ticks, ticks)  # show labels without additional zeros
        ax.axvline(0.05, **kwargsl)
        z_lim_1, z_lim_2 = ax.get_ylim()
        # The flow direction (in-/outflow) was determined from the velocity components.
        ax.text(0.125, (z_min_2 + z_lim_2) / 2, "outflow", **kwargst)
        ax.text(0.125, (z_min_1 + z_min_2) / 2, "inflow", **kwargst)
        ax.text(0.125, (z_min_1 + z_lim_1) / 2, "deep water", **kwargst)
    ax.set_ylabel("depth in m" if letter == "c" else "", size="small")
    ax.yaxis.set_major_formatter((lambda x, pos: f"{-x}") if letter == "c" else "")
    ax.tick_params(labelsize="small")

# Reduce horizontal spacing
fig.get_layout_engine().set(w_pad=0.02*cm, wspace=0.0)

fig.savefig("figures/fig_5_inflow_map.png", dpi=600)